In [1]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn import datasets
from sklearn.tree import DecisionTreeClassifier
import numpy as np
from Testers import Tester
import pandas as pd
import random

seed = 42
np.random.seed(seed)
random.seed(seed)
data = datasets.load_digits()


X = data.data
y = data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True)

In [2]:
from typing import List, Tuple
from Bagging import BaggingModel


def evaluate_fitness(models: List[BaggingModel], X: np.ndarray, y: np.ndarray) -> float:
        predictions_per_model = [model.model.predict(X[:,model.features]) for model in models]
        accuracy_per_model = [accuracy_score(y, pred) for pred in predictions_per_model]
        return np.mean(accuracy_per_model)    

In [3]:
from Bagging import create_bags, create_models, get_accuracy


reps = 10
n = 100
res_n = 5


acc_sum = 0
for r in range(reps):
    X_train_sub, X_test_sub, y_train_sub, y_test_sub = train_test_split(X_train, y_train, test_size=0.2, shuffle=True)
    
    bags_large= create_bags(X=X_train_sub, y=y_train_sub, n_bags=n, with_replacement=True)
    models = create_models(bags_large, n_trees=n)

    predictions_per_model = [model.model.predict(X_test_sub[:,model.features]) for model in models]
    accuracy_per_model = [accuracy_score(y_test_sub, pred) for pred in predictions_per_model]

    sorted_models = sorted(zip(models, accuracy_per_model), key=lambda x: x[1], reverse=True)
    models = [model for model, _ in sorted_models[:res_n]]

    ensemble_accuracy = get_accuracy(X=X_test, y=y_test, models=models)
    acc_sum += ensemble_accuracy
    print(f"[{r}] Accuracy of the large ensemble: {ensemble_accuracy:.4f}")


acc_sum2 = 0
for r in range(reps):
    bags = create_bags(X=X_train, y=y_train, n_bags=res_n, with_replacement=True)
    models = create_models(bags=bags, n_trees=res_n)
    ensemble_accuracy = get_accuracy(X=X_test, y=y_test, models=models)
    acc_sum2 += ensemble_accuracy
    print(f"[{r}] Accuracy of the ensemble: {ensemble_accuracy:.4f}")
    
    
    
print(f"Accuracy of boosted the ensemble: {acc_sum / reps:.4f}")
print(f"Accuracy of the ensemble: {acc_sum2 / reps:.4f}")


[0] Accuracy of the large ensemble: 0.8417
[1] Accuracy of the large ensemble: 0.8389
[2] Accuracy of the large ensemble: 0.8417
[3] Accuracy of the large ensemble: 0.8472
[4] Accuracy of the large ensemble: 0.8361
[5] Accuracy of the large ensemble: 0.8528
[6] Accuracy of the large ensemble: 0.8000
[7] Accuracy of the large ensemble: 0.8556
[8] Accuracy of the large ensemble: 0.8833
[9] Accuracy of the large ensemble: 0.8417
[0] Accuracy of the ensemble: 0.7250
[1] Accuracy of the ensemble: 0.6444
[2] Accuracy of the ensemble: 0.6944
[3] Accuracy of the ensemble: 0.7722
[4] Accuracy of the ensemble: 0.7722
[5] Accuracy of the ensemble: 0.7444
[6] Accuracy of the ensemble: 0.7722
[7] Accuracy of the ensemble: 0.7778
[8] Accuracy of the ensemble: 0.6861
[9] Accuracy of the ensemble: 0.7222
Accuracy of boosted the ensemble: 0.8439
Accuracy of the ensemble: 0.7311
